# 🚢 تجارت‌یار — اجرای سریع روی Google Colab

این دفترچه سامانه‌ی **تجارت‌یار** را در **کمتر از ~۱ دقیقه** روی Colab بالا می‌آورد و یک **لینک عمومی موقت** می‌دهد.

> 🎬 **برای ارائه‌ی زنده** دفترچه‌ی کامل‌تر و مقاوم‌تر `presentation_colab.ipynb` را باز کنید
> (تست سلامت، بازنشانی داده‌ی نمونه و سناریوی ارائه دارد).

**نکته‌ی مهم:** خروجی build (`dist`) از قبل آماده و در مخزن قرار دارد؛ بنابراین **نیازی به `npm install` و `npm run build` نیست**.

**هر سلول را به ترتیب با `Ctrl+Enter` اجرا کنید** (یا `Runtime → Run all`):
1. نصب Node.js (فقط در صورت نیاز)
2. دریافت کد از GitHub
3. اجرای سرور در **حالت ارائه** (`SEED_DEMO=1` → داده‌ی نمونه)
4. بررسی سلامت
5. نمایش فوری داخل Colab
6. دانلود cloudflared
7. راه‌اندازی تونل
8. دریافت لینک دسترسی

In [ ]:
%%bash
# نصب Node.js فقط در صورت نیاز (Colab معمولاً Node ۱۸ یا جدیدتر دارد)
MAJOR=$(node -v 2>/dev/null | sed 's/^v\([0-9]*\).*/\1/')
if [ -n "$MAJOR" ] && [ "$MAJOR" -ge 18 ]; then
  echo "node already present: $(node -v)"
else
  curl -fsSL https://nodejs.org/dist/v20.18.1/node-v20.18.1-linux-x64.tar.xz -o /tmp/node.txz \
    && tar -xJf /tmp/node.txz -C /usr/local --strip-components=1 \
    && echo "node installed: $(node -v)"
fi

In [ ]:
%%bash
# دریافت کد — برای نسخه‌ی دیگر، نام شاخه را در --branch عوض کنید
cd /content && rm -rf Tejaratyarr \
  && git clone --depth 1 --branch main https://github.com/Setayesh-Jafari/Tejaratyarr.git

In [ ]:
%%bash
# اجرای سرور در حالت ارائه (SEED_DEMO=1 → کارتابل با داده‌ی نمونه پر می‌شود)
cd /content/Tejaratyarr || exit 1
pkill -f 'node dist/[s]erver\.cjs' 2>/dev/null; sleep 1   # الگوی [s] تا pkill شلِ خودش را نکشد
SEED_DEMO=1 NODE_ENV=production setsid nohup node dist/server.cjs > /content/server.log 2>&1 < /dev/null &
sleep 1; echo "server starting... (log: /content/server.log)"

In [ ]:
%%bash
# بررسی سلامت + شمارش داده‌ی کارتابل (تا ۳۰ ثانیه تلاش می‌کند)
for i in $(seq 1 30); do
  H=$(curl -s http://localhost:3000/api/health)
  [ -n "$H" ] && break
  sleep 1
done
echo "health : $H"
curl -s http://localhost:3000/api/demo/state; echo
if [ -z "$H" ]; then echo "❌ سرور بالا نیامد — لاگ:"; tail -20 /content/server.log; fi

In [ ]:
from google.colab import output

# نمایش برنامه داخل همین دفترچه (بدون تونل و بدون انتظار)
output.serve_kernel_port_as_window(3000)

In [ ]:
%%bash
cd /content/Tejaratyarr || exit 1
if [ ! -x cloudflared ]; then
  curl -L --progress-bar -o cloudflared https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64
fi
chmod +x cloudflared && ./cloudflared --version

In [ ]:
import subprocess

# بستن هر نمونه‌ی قبلی cloudflared (اگر سلول را دوباره اجرا کنید)
subprocess.run("pkill -f 'cloudflared tunnel' || true", shell=True, capture_output=True)

p = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", "http://localhost:3000",
     "--no-autoupdate", "--logfile", "/content/cloudflared.log"],
    cwd="/content/Tejaratyarr",
    stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL, stdin=subprocess.DEVNULL,
    start_new_session=True,
)
print("cloudflared started (PID", p.pid, ") — برای گرفتن لینک به سلول بعد بروید.")

In [ ]:
import time, re

print("در انتظار لینک تونل (معمولاً ۱۰ تا ۳۰ ثانیه) ...")
url = None
for _ in range(60):
    try:
        m = re.search(r"https://[a-z0-9-]+\.trycloudflare\.com",
                      open("/content/cloudflared.log", errors="ignore").read())
        if m:
            url = m.group(0)
            break
    except FileNotFoundError:
        pass
    time.sleep(2)

print()
if url:
    print("LINK:", url)
    print("(لینک موقت است و تا وقتی این Colab روشن بماند کار می‌کند)")
else:
    print("❌ لینک آماده نشد — همین سلول را دوباره اجرا کنید. آخرین خطوط لاگ:")
    try:
        print(open("/content/cloudflared.log", errors="ignore").read()[-2000:])
    except Exception as e:
        print(e)

## 📌 نکات

- لینک `trycloudflare` **موقت** است و تا زمانی که Colab روشن بماند کار می‌کند.
- **داده‌ی نمونه:** سلول ۳ سرور را با `SEED_DEMO=1` اجرا می‌کند تا کارتابل برای دمو خالی نباشد؛
  در این حالت برچسب «داده نمونه — حالت ارائه» در هدر برنامه دیده می‌شود.
  برای داده‌ی واقعی (کارتابل خالی) همان سلول را بدون `SEED_DEMO=1` اجرا کنید.
- بازنشانی داده‌ی نمونه وسط دمو: `curl -X POST http://localhost:3000/api/demo/seed`
- اگر لینک چاپ نشد: سلول تونل را دوباره اجرا کنید؛ اگر سرور خوابید، سلول ۳ و سپس ۴.
- فعال‌سازی هوش مصنوعی Gemini: قبل از سلول ۳، `GEMINI_API_KEY` را در محیط سرور تنظیم کنید.
- اجرای محلی (با کد منبع): `npm install` سپس `npm run dev` (پورت ۳۰۰۰).
- اجرای تولید محلی: `npm run build` سپس `NODE_ENV=production node dist/server.cjs`.